In [3]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from PIL import Image
import numpy as np
from torchinfo import summary

from torch import nn, optim

import pandas as pd
import os


from tqdm import tqdm 


In [4]:
base_dir = os.path.dirname(os.path.abspath(__name__))


data_root = os.path.join(base_dir, "dataset", "dataset")

train_csv = os.path.join(data_root, "train_solution.csv")
train_img_dir = os.path.join(data_root, "train_images")
test_img_dir = os.path.join(data_root, "test_images")


In [5]:
class FaceDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)
    
    def __getitem__(self, idx):
        img_id = str(self.data_frame.iloc[idx, 0]) 
        img_id += '.jpg'

        img_path = os.path.join(self.img_dir, img_id)

        image = Image.open(img_path).convert('RGB')

        label = int(self.data_frame.iloc[idx, 1])

        if self.transform:
            image = self.transform(image)

        return image, label
    

In [6]:
class FaceTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_names)
    
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        img_id = img_name.replace('.jpg', '')
        return image, img_id

In [7]:

IMG_SIZE = 256
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.3),
    
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])


face_dataset_train = FaceDataset(
    csv_file=train_csv,
    img_dir=train_img_dir,
    transform=transform_train
)

face_dataset_loader = DataLoader(face_dataset_train, batch_size=64, shuffle=True)


In [8]:
transform_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

face_dataset_validation = FaceTestDataset(
    img_dir=test_img_dir,
    transform=transform_valid
)

face_dataset_validation_loader = DataLoader(
    face_dataset_validation,
    batch_size=1,
    shuffle=False 
)

In [9]:
class SkipConnectionBlock(nn.Module):
    def __init__(self, in_channels):
        super(SkipConnectionBlock, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels)
        )
    
    def forward(self, x):
        return torch.relu(self.model(x) + x)

In [10]:
class DeepFakeModel(nn.Module):
    def __init__(self):
        super(DeepFakeModel, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2)
        )

        self.stage1 = nn.Sequential(
            SkipConnectionBlock(32),
            self._transition(32, 64)
        )
        
        self.stage2 = nn.Sequential(
            SkipConnectionBlock(64),
            self._transition(64, 128)
        )
        
        self.stage3 = nn.Sequential(
            SkipConnectionBlock(128),
            self._transition(128, 256)
        )
        
        self.stage4 = nn.Sequential(
            SkipConnectionBlock(256),
            self._transition(256, 512)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 1) 
        )

    def _transition(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.gap(x)

        return self.classifier(x)


In [11]:
model = DeepFakeModel()

num_negatve = 41499
num_positive = 8500
pos_weight_value = torch.tensor([num_negatve/num_positive])

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)


In [12]:
def response(id_img, target):
    data = {
        'id': id_img,
        'target': target
    }

    df = pd.DataFrame(data)
    df.to_csv('result.csv', index=False)

In [15]:
num_epoch = 45

for epoch in range(num_epoch):
    for X, y in tqdm(face_dataset_loader):
        model.train()

        predict = model(X)
        optimizer.zero_grad()
        loss = loss_fn(predict.squeeze(), y.float())
        loss.backward()
        optimizer.step()

model_path = "face_model_weights.pth"
torch.save(model.state_dict(), model_path)

  2%|▏         | 18/782 [01:14<52:23,  4.11s/it] 


KeyboardInterrupt: 

In [ ]:
id_img = []
targets = []

model.eval()
with torch.no_grad():
    for X, names in tqdm(face_dataset_validation_loader):
        out = model(X)
        probabilities = torch.sigmoid(out.squeeze())
        predict = (probabilities > 0.5).int()
        
        id_img.extend(names)
        targets.append(predict.item())

response(id_img, targets)

KeyboardInterrupt: 